In [1]:
import numpy as np
from tqdm import tqdm
import pandas as pd
import json
import time
import re
import requests
from functools import wraps
import subprocess
import utils # Custom python utility functions

In [2]:
data_dir = 'batches_50k'
use_hw = 'cpu'
jvm_mem = '4G'
solr_url = 'http://localhost:8983/solr/test/select'

In [3]:
# Create necessary directories and convert data to javabin format
!mkdir -p {data_dir}

# Convert the data to javabin format with smaller batch size and doc count
# since we're using wikipedia_vector_dump_100.csv.gz which has 100 vectors
!java -cp solr-cuvs-benchmarks-1.0-SNAPSHOT-jar-with-dependencies.jar \
    com.searchscale.benchmarks.Indexer \
    data_file=wikipedia_vector_dump_100.csv.gz \
    output_file={data_dir}/wiki \
    batch_size=50 \
    docs_count=100 \
    legacy=true

# Verify the output file exists
!ls -l {data_dir}/wiki.0

{data_file=wikipedia_vector_dump_100.csv.gz, batch_size=50, legacy=true, docs_count=100, output_file=batches_50k/wiki}
SLF4J(W): No SLF4J providers were found.
SLF4J(W): Defaulting to no-operation (NOP) logger implementation
SLF4J(W): See https://www.slf4j.org/codes.html#noProviders for further details.
batches_50k/wiki.0
batches_50k/wiki.1
-rw-r--r-- 1 root root 512570 Jun 19 21:51 batches_50k/wiki.0


# Configure Solr

In [2]:
# Templates for parameter sweeps
if use_hw == 'cpu':
    model_name = 'hnsw'
    model_params = {
        'dim': 2048,
        'hnswMaxConnections': 32,
        'hnswBeamWidth': 512
    }
elif use_hw == 'gpu':
    model_name = 'cuvs'
    model_params = {
        'dim': 2048,
        'graphDegree': 32,
        'intGraphDegree': 64,
        'cuvsWriterThreads': 8
    }
else:
    raise ValueError('Unknown use_hw value. Choose "cpu" or "gpu".')
    
# Generate solr xml config files
utils.generate_config_xml(model_name, model_params)

# Generate Solr bash scripts
utils.generate_solr_bash_scripts(data_dir, use_hw, jvm_mem)
    
# Start Solr and reset database
subprocess.run("chmod +x *.sh", shell=True, executable="/bin/bash")
print()
subprocess.run("sh ./start_solr_mod.sh", shell=True, executable="/bin/bash")
print()

Generating hnsw schema.xml and solrconfig.xml files.
Completed writing all XML files.
Sucessfully written ./start_solr_mod.sh and ./upload_all_files_mod.sh.

Waiting up to 180 seconds to see Solr running on port 8983 [/]  
Started Solr server on port 8983 (pid=9971). Happy searching!



  adding: schema.xml (deflated 60%)
  adding: solrconfig.xml (deflated 52%)
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  2031  100    60  100  1971     30    998  0:00:02  0:00:01  0:00:01  1028
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0

{
  "responseHeader":{
    "status":0,
    "QTime":160
  }
}{
  "responseHeader":{
    "status":0,
    "QTime":1198
  },
  "success":{
    "localhost:8983_solr":{
      "responseHeader":{
        "status":0,
        "QTime":810
      },
      "core":"test_shard1_replica_n1"
    }
  }
}


100   226  100   226    0     0    188      0  0:00:01  0:00:01 --:--:--   188


In [3]:
# Start javabin upload and indexing pipeline
start_time = time.perf_counter()
subprocess.run("sh ./upload_all_files_mod.sh", shell=True, executable="/bin/bash")
indexing_time = time.perf_counter() - start_time

print('Indexing time:', f'{indexing_time:.2f} seconds')

Uploading batches_50k/wiki.0...
{
  "responseHeader":{
    "rf":1,
    "status":0,
    "QTime":129093
  }
}All files in the directory uploaded.
Indexing time: 129.46 seconds


# Vector Search

In [4]:
# Specify search parameters:
topK = 5  # Reduced since we have a smaller dataset
return_limit = topK
num_queries = 20  # Reduced since we only have 100 vectors total

if use_hw == 'cpu':
    # Build params: hnswBeamWidth=efConstruction, hnswMaxConnections=M.
    # No efSearch parameter. Use default value.
    query_prefix = f'{{!knn f=article_vector topK={topK}}}'
elif use_hw == 'gpu':
    cagraITopK = 10
    cagraSearchWidth = 32
    query_prefix = f'{{!cuvs f=article_vector cagraITopK={cagraITopK} cagraSearchWidth={cagraSearchWidth} topK={topK}}}'
else:
    raise ValueError('Unknown use_hw value. Choose "cpu" or "gpu".')


# Load query vectors
ids, query_vector_strs = utils.load_query_vectors(f'{data_dir}/wiki.0', num_queries)

SLF4J(W): No SLF4J providers were found.
SLF4J(W): Defaulting to no-operation (NOP) logger implementation
SLF4J(W): See https://www.slf4j.org/codes.html#noProviders for further details.


Successfully loaded query vectors.


In [5]:
def time_it(func: any):
    """returns result and elapsed time"""
    @wraps(func)
    def inner(*args, **kwargs):
        pref = time.perf_counter()
        result = func(*args, **kwargs)
        delta = time.perf_counter() - pref
        return result, delta
    return(inner)

@time_it
def run_single_query(query_prefix, vector_str, return_limit=10):
    """
    Function for submitting individual queries to Solr.
    """
    
    query_obj = {
      "query": {
        "lucene": {
          "df": "name",
          "query": query_prefix + vector_str
        }
      },
      "fields": "id",
      "limit": return_limit
    }

    response = requests.post(solr_url, json = query_obj)

    # Convert response text to dict:
    response = json.loads(response.text)

    # Initialize outputs
    out = {}

    # Validate responses
    if response['responseHeader']['status'] == 0:
        out['QTime'] = response['responseHeader']['QTime']
    else:
        raise ValueError('Query did not complete successfully.')

    # Accumulate doc_ids
    out['doc_ids'] = [d['id'] for d in response['response']['docs']]
    return(out)

def run_all_queries(query_prefix, query_vector_strs, batch_size=1):
    print(f'batch_size={batch_size}, num_queries={num_queries}')
    print(model_params)
    start_time = time.perf_counter()
    query_results = [run_single_query(query_prefix, vector_str) for vector_str in tqdm(query_vector_strs, desc='Run progress')]
    elapsed_time = time.perf_counter() - start_time

    # Store matches
    topK_ids = [r[0] for r in query_results]
    topK_ids = [[int(ii) for ii in topK_ids[jj]['doc_ids']] for jj in range(len(query_results))]
    
    # Assemble run times
    timing_store = [r[1] for r in query_results]
    run_times = pd.Series(timing_store)
    P99 = run_times.quantile(.99)
    
    print('Wall time:', f'{elapsed_time:.2f} seconds')
    print(f'QPS={num_queries/elapsed_time:.2f}, P99={P99*1000:0.2f} ms')
    print()
    return(topK_ids)

topK_ids = run_all_queries(query_prefix, query_vector_strs)

batch_size=1, num_queries=1024
{'dim': 2048, 'hnswMaxConnections': 32, 'hnswBeamWidth': 512}


Run progress: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1024/1024 [00:05<00:00, 191.76it/s]

Wall time: 5.35 seconds
QPS=191.53, P99=9.67 ms



In [6]:
topK_ids[0]

[39, 42986, 108115, 16343, 68915, 7517, 110736, 74969, 110319, 67337]

# Generate Ground Truth and Calculate Recall

In [4]:
# Specify search parameters:
topK = 10
return_limit = topK
num_queries = 20

if use_hw == 'cpu':
    # Build params: hnswBeamWidth=efConstruction, hnswMaxConnections=M.
    # No efSearch parameter. Use default value.
    query_prefix = f'{{!knn f=article_vector topK={topK}}}'
elif use_hw == 'gpu':
    cagraITopK = 10
    cagraSearchWidth = 32
    query_prefix = f'{{!cuvs f=article_vector cagraITopK={cagraITopK} cagraSearchWidth={cagraSearchWidth} topK={topK}}}'
else:
    raise ValueError('Unknown use_hw value. Choose "cpu" or "gpu".')


# Load query vectors
ids, query_vector_strs = utils.load_query_vectors(f'{data_dir}/wiki.0', num_queries)

Successfully loaded query vectors.


SLF4J(W): No SLF4J providers were found.
SLF4J(W): Defaulting to no-operation (NOP) logger implementation
SLF4J(W): See https://www.slf4j.org/codes.html#noProviders for further details.


In [5]:
# Load query vectors
ids, query_vector_strs = utils.load_query_vectors(f'{data_dir}/wiki.0', num_queries)

Skiping startJVM. JVM already running.
Successfully loaded query vectors.


In [6]:
# Convert query vectors from strings back to numpy arrays
query_vectors = []

for vec_str in query_vector_strs:
    # Remove brackets and split by comma
    values = vec_str.strip('[]').split(',')
    # Convert to float array
    vec = np.array([float(x) for x in values])
    query_vectors.append(vec)
query_vectors = np.stack(query_vectors)

# Load all vectors from the dataset to generate ground truth
print("Loading dataset vectors...")
dataset_ids, dataset_vector_strs = utils.load_query_vectors(f'{data_dir}/wiki.0', 50)  
dataset_vectors = []
for vec_str in dataset_vector_strs:
    values = vec_str.strip('[]').split(',')
    vec = np.array([float(x) for x in values])
    dataset_vectors.append(vec)
dataset_vectors = np.stack(dataset_vectors)

# Generate ground truth
print("\nGenerating ground truth...")
distances, ground_truth_indices = utils.calc_truth(
    dataset=dataset_vectors,
    queries=query_vectors,
    k=topK,
    metric="cosine"  # Match the similarity function used in Solr config
)

Loading dataset vectors...
Skiping startJVM. JVM already running.
Successfully loaded query vectors.

Generating ground truth...
Building index for full dataset (50 vectors)...
Searching with full dataset...


In [7]:
dataset_ids

[39,
 303,
 307,
 308,
 309,
 316,
 330,
 332,
 334,
 336,
 339,
 340,
 344,
 359,
 569,
 580,
 586,
 595,
 597,
 599,
 600,
 612,
 615,
 620,
 621,
 624,
 627,
 628,
 634,
 642,
 649,
 651,
 653,
 656,
 657,
 662,
 663,
 665,
 666,
 670,
 674,
 675,
 676,
 677,
 680,
 681,
 682,
 689,
 690,
 691]

In [8]:
ground_truth_indices

array([[ 0, 44, 33, 28, 26, 49, 46, 37, 41,  9],
       [ 1, 25, 18, 19, 48, 20, 45, 46, 47,  7],
       [ 2, 11, 42, 10,  4, 35, 22, 49, 27, 46],
       [ 3, 11, 46, 23, 27, 45, 12, 33,  7, 42],
       [ 4, 17, 22, 11,  2, 35, 10, 31, 27, 46],
       [ 5, 48, 18, 40, 31, 47,  7,  8, 25, 19],
       [ 6, 18, 12, 19, 32,  7, 16, 25, 39,  1],
       [ 7, 19, 18, 12, 25, 27, 17,  1, 48, 20],
       [ 8, 32, 16, 36, 18, 13, 47, 28, 46, 39],
       [ 9, 43, 28, 41,  0, 26, 30, 33, 39, 29],
       [10, 42, 11, 33, 17, 44,  2, 12, 46,  4],
       [11,  3, 46, 27, 17, 42,  4,  2, 10, 12],
       [12,  7, 27, 11, 18, 19, 17,  6, 25,  3],
       [13,  8, 36, 12, 16, 39,  6, 32, 10,  7],
       [14, 40,  8, 47,  5, 32, 15, 18, 48, 13],
       [15,  8, 13, 36,  5, 16, 14, 40, 39, 32],
       [16, 39,  8, 32, 36, 13, 28,  6, 18, 25],
       [17,  4, 11, 27,  7, 22, 10, 12, 35, 31],
       [18, 19,  1, 25, 48, 47,  7, 45, 32, 46],
       [19, 25, 18,  1, 48,  7, 20, 46, 45, 32]])

In [9]:
# Create synthetic topK_ids for testing
import numpy as np

# Get the dataset size (number of vectors)
dataset_size = len(dataset_ids)  # This should be 100 for your case

# Create random indices for each query as regular Python lists
# Shape: (num_queries, topK)
topK_ids = [
    list(map(int, np.random.choice(dataset_size, size=topK, replace=False)))
    for _ in range(num_queries)
]

print(topK_ids)

[[32, 37, 36, 1, 44, 24, 33, 27, 26, 10], [21, 35, 44, 16, 40, 39, 18, 37, 22, 48], [10, 32, 46, 39, 38, 47, 7, 20, 19, 36], [1, 6, 43, 9, 46, 22, 42, 45, 37, 7], [9, 13, 20, 14, 32, 4, 23, 25, 35, 42], [30, 41, 24, 8, 3, 48, 45, 4, 6, 26], [49, 4, 1, 24, 33, 15, 35, 38, 13, 41], [47, 8, 12, 16, 41, 0, 34, 37, 22, 23], [14, 15, 27, 7, 32, 18, 8, 49, 25, 43], [4, 15, 6, 37, 44, 38, 36, 46, 7, 28], [40, 37, 29, 46, 1, 12, 13, 33, 24, 41], [0, 12, 1, 46, 8, 49, 18, 20, 10, 38], [11, 28, 15, 29, 32, 2, 19, 5, 35, 42], [6, 7, 33, 34, 39, 12, 16, 35, 44, 37], [20, 8, 17, 37, 23, 35, 22, 25, 14, 16], [47, 30, 49, 33, 35, 17, 5, 26, 36, 7], [35, 33, 14, 30, 32, 6, 41, 45, 2, 23], [49, 34, 6, 36, 16, 27, 41, 33, 37, 5], [11, 18, 7, 32, 27, 16, 19, 41, 0, 49], [39, 2, 4, 45, 12, 24, 46, 1, 44, 40]]


In [10]:
# Calculate recall
print("Calculating recall...")

# Convert topK_ids and ground_truth_indices to numpy arrays
topK_array = np.array(topK_ids)
ground_truth_array = np.array(ground_truth_indices)

# Calculate recall using the utility function
recall = utils.calc_recall(topK_array, ground_truth_array)

print(f"\nRecall@{topK} = {recall:.4f}")

# Print some example comparisons
print("\nExample comparisons (first 3 queries):")
for i in range(3):
    print(f"\nQuery {i}:")
    print(f"Ground Truth IDs: {ground_truth_indices[i]}")
    print(f"Retrieved IDs:    {topK_ids[i]}")
    # Calculate intersection
    intersection = set(ground_truth_indices[i]) & set(topK_ids[i])
    print(f"Common IDs:       {sorted(list(intersection))}")
    print(f"Recall:          {len(intersection)/len(ground_truth_indices[i]):.4f}")


Calculating recall...

Recall@10 = 0.2450

Example comparisons (first 3 queries):

Query 0:
Ground Truth IDs: [ 0 44 33 28 26 49 46 37 41  9]
Retrieved IDs:    [32, 37, 36, 1, 44, 24, 33, 27, 26, 10]
Common IDs:       [26, 33, 37, 44]
Recall:          0.4000

Query 1:
Ground Truth IDs: [ 1 25 18 19 48 20 45 46 47  7]
Retrieved IDs:    [21, 35, 44, 16, 40, 39, 18, 37, 22, 48]
Common IDs:       [18, 48]
Recall:          0.2000

Query 2:
Ground Truth IDs: [ 2 11 42 10  4 35 22 49 27 46]
Retrieved IDs:    [10, 32, 46, 39, 38, 47, 7, 20, 19, 36]
Common IDs:       [10, 46]
Recall:          0.2000
